# Gas Turbine NOx Emission Prediction (regression)

Predict nitrogen-oxide (NOx) emissions of a gas turbine from sensor readings.

| | |
|---|---|
| **Task** | Regression (target: `NOX`) |
| **Data** | UCI dataset #551: 36,733 records (2011-2015), 11 features: `year`, `AT`, `AP`, `AH`, `AFDP`, `GTEP`, `TIT`, `TAT`, `TEY`, `CDP`, `CO` |
| **Model** | Random Forest Regressor (150 trees, `min_samples_leaf=4`, `min_samples_split=4`, `ccp_alpha=0.02`) |
| **Result** | **R² = 0.883**, MSE = 15.24 (RMSE ≈ 3.9) on the test set |

## Approach
- 80/20 random split (`random_state=13`).
- Random Forest baseline, then `RandomizedSearchCV` (10 candidates, 3-fold) over the main hyperparameters.

## Results

| | R² | MSE |
|---|---:|---:|
| Train | 0.914 | 11.87 |
| Test | 0.883 | 15.24 |

For context, the standard deviation of NOx is 11.7, so the typical error (RMSE ≈ 3.9) is roughly a third of that. The search's best cross-validated MSE was 14.72, close to the test MSE, so the original model was already near the best configuration found.

## Limitations & next steps
- 6 of the 30 cross-validation fits failed because `max_features='auto'` is no longer accepted by recent scikit-learn versions. Replace it (e.g. with `1.0`) and re-run so the notebook has no warnings.
- The final test numbers are from the original model, not a refit with the search's best parameters.
- The records are time-ordered (`year` is a feature), so a random split may be optimistic. Compare with a split by year.
- `CO` is itself an emission measurement; check whether it would be available when the model is used in practice.
- Next: gradient boosting (XGBoost / LightGBM), feature importance, and residual plots.


In [2]:
!pip install ucimlrepo

In [3]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score


In [4]:

# fetch dataset
gas_turbine_co_and_nox_emission_data_set = fetch_ucirepo(id=551)

# data (as pandas dataframes)
X = gas_turbine_co_and_nox_emission_data_set.data.features
y = gas_turbine_co_and_nox_emission_data_set.data.targets

# metadata
print(gas_turbine_co_and_nox_emission_data_set.metadata)

# variable information
print(gas_turbine_co_and_nox_emission_data_set.variables)


{'uci_id': 551, 'name': 'Gas Turbine CO and NOx Emission Data Set', 'repository_url': 'https://archive.ics.uci.edu/dataset/551/gas+turbine+co+and+nox+emission+data+set', 'data_url': 'https://archive.ics.uci.edu/static/public/551/data.csv', 'abstract': 'The dataset contains 36733 instances of 11 sensor measures aggregated over one hour, from a gas turbine located in Turkey for the purpose of studying flue gas emissions, namely CO and NOx.', 'area': 'Computer Science', 'tasks': ['Regression', 'Clustering'], 'characteristics': ['Multivariate'], 'num_instances': 36733, 'num_features': 12, 'feature_types': ['Real'], 'demographics': [], 'target_col': None, 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2019, 'last_updated': 'Fri Mar 29 2024', 'dataset_doi': '10.24432/C5WC95', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "The dataset contains 36733 instances of 11 sensor measures aggregated over one hour (by mea

In [5]:
df = pd.concat([X,y],axis=1)
df

,year,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX
0,2011,4.5878,1018.7,83.675,3.5758,23.979,1086.2,549.83,134.67,11.898,0.32663,81.952
1,2011,4.2932,1018.3,84.235,3.5709,23.951,1086.1,550.05,134.67,11.892,0.44784,82.377
2,2011,3.9045,1018.4,84.858,3.5828,23.990,1086.5,550.19,135.10,12.042,0.45144,83.776
3,2011,3.7436,1018.3,85.434,3.5808,23.911,1086.5,550.17,135.03,11.990,0.23107,82.505
4,2011,3.7516,1017.8,85.182,3.5781,23.917,1085.9,550.00,134.67,11.910,0.26747,82.028
...,...,...,...,...,...,...,...,...,...,...,...,...
36728,2015,3.6268,1028.5,93.200,3.1661,19.087,1037.0,541.59,109.08,10.411,10.99300,89.172
36729,2015,4.1674,1028.6,94.036,3.1923,19.016,1037.6,542.28,108.79,10.344,11.14400,88.849
36730,2015,5.4820,1028.5,95.219,3.3128,18.857,1038.0,543.48,107.81,10.462,11.41400,96.147
36731,2015,5.8837,1028.7,94.200,3.9831,23.563,1076.9,550.11,131.41,11.771,3.31340,64.738


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36733 entries, 0 to 36732
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    36733 non-null  int64  
 1   AT      36733 non-null  float64
 2   AP      36733 non-null  float64
 3   AH      36733 non-null  float64
 4   AFDP    36733 non-null  float64
 5   GTEP    36733 non-null  float64
 6   TIT     36733 non-null  float64
 7   TAT     36733 non-null  float64
 8   TEY     36733 non-null  float64
 9   CDP     36733 non-null  float64
 10  CO      36733 non-null  float64
 11  NOX     36733 non-null  float64
dtypes: float64(11), int64(1)
memory usage: 3.4 MB


In [7]:
df. describe(

)

,year,AT,AP,AH,AFDP,GTEP,TIT,TAT,TEY,CDP,CO,NOX
count,36733.000000,36733.000000,36733.000000,36733.000000,36733.000000,36733.000000,36733.000000,36733.000000,36733.000000,36733.000000,36733.000000,36733.000000
mean,2012.985735,17.712726,1013.070165,77.867015,3.925518,25.563801,1081.428084,546.158517,133.506404,12.060525,2.372468,65.293067
std,1.418965,7.447451,6.463346,14.461355,0.773936,4.195957,17.536373,6.842360,15.618634,1.088795,2.262672,11.678357
min,2011.000000,-6.234800,985.850000,24.085000,2.087400,17.698000,1000.800000,511.040000,100.020000,9.851800,0.000388,25.905000
25%,2012.000000,11.781000,1008.800000,68.188000,3.355600,23.129000,1071.800000,544.720000,124.450000,11.435000,1.182400,57.162000
50%,2013.000000,17.801000,1012.600000,80.470000,3.937700,25.104000,1085.900000,549.880000,133.730000,11.965000,1.713500,63.849000
75%,2014.000000,23.665000,1017.000000,89.376000,4.376900,29.061000,1097.000000,550.040000,144.080000,12.855000,2.842900,71.548000
max,2015.000000,37.103000,1036.600000,100.200000,7.610600,40.716000,1100.900000,550.610000,179.500000,15.159000,44.103000,119.910000


In [8]:
x=df.drop('NOX',axis=1)
y =df['NOX']


In [9]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=13)

In [10]:
model = RandomForestRegressor(n_estimators=150,random_state=13,ccp_alpha=0.02,max_features=30,min_samples_leaf=4,min_samples_split=4)
model.fit(x_train,y_train)

RandomForestRegressor(ccp_alpha=0.02, max_features=30, min_samples_leaf=4,
                      min_samples_split=4, n_estimators=150, random_state=13)

In [11]:
y_pred_train = model.predict(x_train)
y_pred_test = model.predict(x_test)


In [12]:



print('mse for train',mean_squared_error(y_train,y_pred_train))
print('r1 for train',r2_score(y_train,y_pred_train))

mse for train 11.867893726128282
r1 for train 0.9139287104969955


In [13]:
print('mse for test',mean_squared_error(y_test,y_pred_test))
print('r1 for test',r2_score(y_test,y_pred_test))
#

mse for test 15.239525238729613
r1 for test 0.8830918003838676


In [15]:
from sklearn.model_selection import RandomizedSearchCV

# Define the parameter grid based on the provided parameters as a starting point
param_grid = {
    'n_estimators': [100, 150, 200],
    'max_features': [20, 30, 'auto'], # 'auto' is equivalent to n_features for RandomForestRegressor
    'min_samples_leaf': [2, 4, 6],
    'min_samples_split': [2, 4, 6],
    'ccp_alpha': [0.01, 0.02, 0.03]
}

# Initialize the RandomForestRegressor model
rf = RandomForestRegressor(random_state=13)

# Initialize RandomizedSearchCV
grid_search = RandomizedSearchCV(estimator=rf, param_distributions=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fit the grid search to the data
grid_search.fit(x_train, y_train)

# Print the best parameters and best score
print("Best parameters found: ", grid_search.best_params_)
print("Best negative mean squared error found: ", grid_search.best_score_)

print('mse for train',mean_squared_error(y_train,y_pred_train))
print('r1 for train',r2_score(y_train,y_pred_train))
#
print('mse for test',mean_squared_error(y_test,y_pred_test))
print('r1 for test',r2_score(y_test,y_pred_test))


Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
6 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
6 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/sklearn/base.py", line 436, in _validate_params
    validate_param

Best parameters found:  {'n_estimators': 150, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 30, 'ccp_alpha': 0.01}
Best negative mean squared error found:  -14.715834816963005
mse for train 11.867893726128282
r1 for train 0.9139287104969955
mse for test 15.239525238729613
r1 for test 0.8830918003838676
